# LLM Efficiency Frontier — analysis

Production-engineering framework that turns **established LLM scaling laws** (power-law / Chinchilla) into **resource and energy allocation decisions** under joint cost and carbon constraints.

**Scope note.** The empirical engine is the power law. The reciprocal-log form is a theoretically motivated *conservative bound* (thermodynamic/material limits), **not** a superior fit — over the observed range it does not outperform the power law.

**Data.** Reconstructed Hoffmann et al. (2022) points from Besiroglu et al. (2024), Epoch AI `epoch-research/analyzing-chinchilla` (245 points).

In [ ]:
import sys; sys.path.append('../src')
import numpy as np, pandas as pd, json
import analysis as A
A.DATA='../data/llm_scaling_dataset.csv'
N,D,L = A.load()
print('points:', len(N), '| N range:', f'{N.min():.2e}', '-', f'{N.max():.2e}')

## 1. Full Chinchilla law L(N,D) — answers the (N, D, C) critique with real data

In [ ]:
p = A.fit_ND(N,D,L)
pred = A.L_ND(N,D,p); r2 = 1-np.sum((L-pred)**2)/np.sum((L-L.mean())**2)
print('E=%.3f A=%.0f alpha=%.3f B=%.0f beta=%.3f | R2=%.4f' % (p['E'],p['A'],p['alpha'],p['B'],p['beta'],r2))

## 2. Compute-optimal frontier  N*(C) = argmin_N L(N, C/6N)

In [ ]:
Cs = np.logspace(18,22,60)
Nf,Lf = A.frontier(p,Cs)
for x in [1e8,1e9,1e10]: print(f'L*(N={x:.0e}) = {np.interp(x,Nf,Lf):.3f}')

## 3. Marginal gain per doubling & efficiency threshold by hurdle rate

In [ ]:
ns = np.logspace(8,10,400); g = A.gain_curve(Nf,Lf,ns)
for h in [0.02,0.04,0.06,0.08]:
    t = A.threshold(ns,g,h)
    print(f'hurdle {int(h*100)}%  ->  N* = {t:.2e}  (log10={np.log10(t):.2f})')
print(f'gain @1e9 = {np.interp(1e9,ns,g)*100:.2f}% per doubling')

## 4. Reciprocal-log vs power-law on the frontier (honest comparison)

In [ ]:
from scipy.optimize import curve_fit
pl,_ = curve_fit(A.log_model,Nf,Lf,p0=[15,-5,2.0],bounds=([0,-8,1.0],[60,5,2.4]),maxfev=200000)
pp,_ = curve_fit(A.power_law,Nf,Lf,p0=[400,0.34,1.7],maxfev=200000)
print('log  :', A.gof(A.log_model,pl,Nf,Lf,3))
print('power:', A.gof(A.power_law,pp,Nf,Lf,3))
print('\nThe power law tracks the frontier; the reciprocal-log does not win in-range.')

## 5. Figures
All four figures are regenerated by `python ../src/analysis.py` into `../figures/`.